# HurricaneMap Data Analysis Starter Notebook

**Interactive notebook for exploring U.S. hurricane landfalls using HurricaneMap's NOAA HURDAT2 dataset.**

This notebook demonstrates how to load, filter, and analyze 174 years of hurricane landfall data. You can:
- Filter by year range, Saffir-Simpson category, or state
- Compute climatology (trends, averages, extremes)
- Create visualizations (time series, histograms, maps)
- Export analysis results

**Run online (no installation required):** [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SysAdminDoc/HurricaneMap/blob/main/notebooks/analysis-starter.ipynb)

**Or run locally:** Clone the HurricaneMap repo and install dependencies:
```bash
git clone https://github.com/SysAdminDoc/HurricaneMap.git
cd HurricaneMap
pip install pandas numpy matplotlib
jupyter notebook notebooks/analysis-starter.ipynb
```


## Setup

Load required libraries and data files.

In [ ]:
import pandas as pd
import json
import numpy as np
import matplotlib.pyplot as plt
import os
from pathlib import Path
from datetime import datetime, timezone
import hashlib

# The release check supplies these paths so execution never writes into the checkout.
NOTEBOOK_ROOT = Path(os.environ.get('HURRICANEMAP_NOTEBOOK_ROOT', Path.cwd()))
DATA_DIR = NOTEBOOK_ROOT / 'data'
OUTPUT_DIR = Path(os.environ.get('HURRICANEMAP_NOTEBOOK_OUTPUT', Path.cwd()))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# For Google Colab: uncomment to download data files
# !mkdir -p data
# !wget -q -O data/landfalls.json https://raw.githubusercontent.com/SysAdminDoc/HurricaneMap/main/data/landfalls.json
# !wget -q -O data/storms.json https://raw.githubusercontent.com/SysAdminDoc/HurricaneMap/main/data/storms.json
# !wget -q -O data/metadata.json https://raw.githubusercontent.com/SysAdminDoc/HurricaneMap/main/data/metadata.json
# !wget -q -O data/release-manifest.json https://raw.githubusercontent.com/SysAdminDoc/HurricaneMap/main/data/release-manifest.json
# !wget -q -O data/impacts.json https://raw.githubusercontent.com/SysAdminDoc/HurricaneMap/main/data/impacts.json

# Load the landfall dataset
with (DATA_DIR / 'landfalls.json').open(encoding='utf-8') as f:
    landfalls_raw = json.load(f)

# Load storms for additional metadata
with (DATA_DIR / 'storms.json').open(encoding='utf-8') as f:
    storms_raw = json.load(f)

# Keep analysis outputs tied to the exact checked-in release.
with (DATA_DIR / 'metadata.json').open(encoding='utf-8') as f:
    metadata_raw = json.load(f)
with (DATA_DIR / 'release-manifest.json').open(encoding='utf-8') as f:
    release_manifest_raw = json.load(f)

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def build_citations(access_date=None):
    access_date = access_date or datetime.now(timezone.utc).date().isoformat()
    artifacts = {record['path']: record for record in release_manifest_raw['artifacts']}
    atlantic = artifacts['data/hurdat2-atlantic.txt']
    nepac = artifacts['data/hurdat2-nepac.txt']
    app_version = metadata_raw['generator']['app_version']
    release_pin = sha256_file(DATA_DIR / 'release-manifest.json')
    citation_url = f'https://sysadmindoc.github.io/HurricaneMap/#v=1&rel={release_pin}'
    revision_date = atlantic['source_date'] if atlantic['source_date'] == nepac['source_date'] else f"{atlantic['source_date']}, {nepac['source_date']}"
    source_hashes = f"Atlantic SHA-256: {atlantic['sha256']}; Eastern Pacific SHA-256: {nepac['sha256']}"
    year = access_date[:4]
    apa = f"SysAdminDoc. ({year}). HurricaneMap: Interactive hurricane landfall atlas (version {app_version}) [Data set and web application]. HURDAT2 revision {revision_date}; {source_hashes}. Retrieved {access_date}, from {citation_url}"
    bibtex = f"@software{{hurricanemap_{year},\n  author = {{Parker, Matt}},\n  title = {{HurricaneMap: Interactive hurricane landfall atlas}},\n  year = {{{year}}},\n  version = {{{app_version}}},\n  url = {{{citation_url}}},\n  note = {{HURDAT2 revision {revision_date}; {source_hashes}; accessed {access_date}; source URLs: {atlantic['source_url']}, {nepac['source_url']}}}\n}}"
    return {'apa': apa, 'bibtex': bibtex, 'url': citation_url}

citation = build_citations()
APA_CITATION = citation['apa']
BIBTEX_CITATION = citation['bibtex']
print('\nAPA citation:\n' + APA_CITATION)
print('\nBibTeX citation:\n' + BIBTEX_CITATION)

# Convert to pandas DataFrame
df = pd.DataFrame(landfalls_raw)
df['month'] = pd.to_datetime(df['t'], utc=True).dt.month

print(f"Loaded {len(df)} landfall records from {df['year'].min()}-{df['year'].max()}")
print(f"Data covers {df['storm_id'].nunique()} unique storms")
print(f"Release generated at {metadata_raw['generated_at_utc']} from {metadata_raw['generator']['source_commit'][:12]}")
print(f"\nFirst few records:")
df.head()

## Data Overview

Understand the structure and content of the dataset.

In [ ]:
# Check column data types and missing values
print("Dataset shape:", df.shape)
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())
print("\nBasic statistics:")
df.describe()

## Filtering & Subsetting

Filter the data by year range, category, or location.

In [ ]:
# Filter by year range
df_recent = df[(df['year'] >= 1980) & (df['year'] <= 2025)]
print(f"Landfalls 1980-2025: {len(df_recent)} records, {df_recent['storm_id'].nunique()} storms")

# Filter by category (3, 4, 5 = major hurricanes)
df_major = df[df['category'] >= 3]
print(f"\nMajor hurricanes (Cat 3-5): {len(df_major)} landfalls, {df_major['storm_id'].nunique()} storms")

# Filter by state
df_florida = df[df['state'] == 'Florida']
print(f"\nFlorida landfalls: {len(df_florida)} records, {df_florida['storm_id'].nunique()} storms")

# Combine filters: recent major hurricanes in Florida
df_subset = df[(df['year'] >= 1980) & (df['category'] >= 3) & (df['state'] == 'Florida')]
print(f"\nRecent (1980+) major hurricanes in Florida: {len(df_subset)} records")
print(df_subset[['year', 'name', 'category', 'wind', 'state']])

## Climatology & Trends

Compute basic statistics and trends over time.

In [ ]:
# Landfalls per year
landfalls_per_year = df.groupby('year').size()
print(f"Average landfalls per year (all time): {landfalls_per_year.mean():.2f}")
print(f"Busiest year: {landfalls_per_year.idxmax()} ({landfalls_per_year.max()} landfalls)")

# Strongest storm at landfall
strongest = df.loc[df['wind'].idxmax()]
print(f"\nStrongest at landfall: {strongest['name']} ({strongest['year']})")
print(f"  Wind: {strongest['wind']} kt ({strongest['wind'] * 1.15:.0f} mph)")
print(f"  Category: {strongest['category']}")
print(f"  Location: {strongest['state']}")

# Category distribution
print("\nLandfalls by category:")
cat_counts = df['category'].value_counts().sort_index()
for cat, count in cat_counts.items():
    pct = count / len(df) * 100
    cat_name = 'TS' if cat <= 0 else f'Cat {cat}'
    print(f"  {cat_name}: {count} ({pct:.1f}%)")

## Geographic Analysis

Analyze landfalls by state and region.

In [ ]:
# Landfalls by state
state_counts = df['state'].value_counts().head(10)
print("Top 10 states by landfall count:")
print(state_counts)

# Visualize as bar chart
plt.figure(figsize=(12, 6))
state_counts.plot(kind='bar', color='steelblue')
plt.title('U.S. Landfalls by State (Top 10)')
plt.xlabel('State')
plt.ylabel('Number of Landfalls')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Major hurricanes by state
major_by_state = df[df['category'] >= 3]['state'].value_counts().head(10)
print("\nTop 10 states by major hurricane (Cat 3-5) landfalls:")
print(major_by_state)

## Time Series Analysis

Plot trends in landfall frequency and intensity over time.

In [ ]:
# Annual landfall count over time
annual_counts = df.groupby('year').size()

plt.figure(figsize=(14, 5))
plt.bar(annual_counts.index, annual_counts.values, alpha=0.6, color='steelblue')

# Add 10-year rolling average
rolling_avg = annual_counts.rolling(window=10).mean()
plt.plot(rolling_avg.index, rolling_avg.values, color='red', linewidth=2, label='10-year average')

plt.xlabel('Year')
plt.ylabel('Number of Landfalls')
plt.title('U.S. Landfalls Over Time (1851-2025)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Average wind speed at landfall over time (5-year rolling avg)
df['year_windspeed'] = df['wind']
avg_wind_by_year = df.groupby('year')['wind'].mean()
rolling_wind = avg_wind_by_year.rolling(window=5).mean()

plt.figure(figsize=(14, 5))
plt.scatter(avg_wind_by_year.index, avg_wind_by_year.values, alpha=0.4, s=30, label='Annual average')
plt.plot(rolling_wind.index, rolling_wind.values, color='red', linewidth=2, label='5-year average')
plt.xlabel('Year')
plt.ylabel('Wind Speed at Landfall (kt)')
plt.title('Average Wind Speed at Landfall Over Time')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Seasonal Patterns

Analyze when landfalls typically occur.

In [ ]:
# Landfalls by month
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
monthly_counts = df['month'].value_counts().sort_index()

plt.figure(figsize=(12, 5))
plt.bar(monthly_counts.index, monthly_counts.values, color='coral')
plt.xlabel('Month')
plt.ylabel('Number of Landfalls')
plt.title('U.S. Landfalls by Month (1851-2025)')
plt.xticks(range(1, 13), month_names)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print("Peak landfall month: August-October")
print(f"Aug-Sep-Oct landfalls: {df[df['month'].isin([8, 9, 10])].shape[0]} / {len(df)} ({df[df['month'].isin([8, 9, 10])].shape[0]/len(df)*100:.1f}%)")

## Export Results

Save your analysis to CSV for further use.

In [ ]:
# Export filtered dataset
major_hurricanes_path = OUTPUT_DIR / 'major_hurricanes_landfalls.csv'
df_major.to_csv(major_hurricanes_path, index=False)
print(f"Exported {len(df_major)} major hurricane landfalls to '{major_hurricanes_path}'")

# Export summary statistics
summary_stats = {
    'Total Landfalls': len(df),
    'Unique Storms': df['storm_id'].nunique(),
    'Year Range': f"{df['year'].min()}-{df['year'].max()}",
    'States': df['state'].nunique(),
    'Major Hurricanes': len(df[df['category'] >= 3]),
    'Average Wind': f"{df['wind'].mean():.1f} kt",
    'Strongest Wind': f"{df['wind'].max():.0f} kt",
}
print("\nSummary Statistics:")
for key, value in summary_stats.items():
    print(f"  {key}: {value}")

## Next Steps

- **Explore specific storms:** Filter by name (e.g., `df[df['name'] == 'KATRINA']`)
- **Spatial analysis:** Plot landfalls on a map using `folium` or `plotly`
- **Machine learning:** Use clustering or regression to predict landfall patterns
- **Combine with other data:** Merge with impacts data (deaths, damages) for socioeconomic analysis
- **Share findings:** Publish results to GitHub or academic repositories

## Resources

- **HurricaneMap GitHub:** https://github.com/SysAdminDoc/HurricaneMap
- **Data sources:** See [LICENSE.md](https://github.com/SysAdminDoc/HurricaneMap/blob/main/LICENSE.md) for full attribution
- **NOAA HURDAT2:** https://www.nhc.noaa.gov/data/hurdat/
- **Pandas documentation:** https://pandas.pydata.org/docs/
- **Matplotlib:** https://matplotlib.org/stable/contents.html
